In [6]:
import pandas as pd
import re
import json

In [7]:
def extract_first_json(text):
    
    text = (
        text
        .replace(" ","")
        .replace("\n","")
        .replace("True", "true")
        .replace("False", "false")
        .replace("None", "null")
        .replace("'",'"')
    )
    match = re.search(r'\{.*?\}', text)
    if match:
        try:
            json_data = json.loads(match.group())
            return json_data
        except json.JSONDecodeError as e:
            print(match.group())
            print("failed", e)
            return {}
    else:
        print("format failed")
        return {}

def extract_last_json(text):
    text = (
        text
        .replace(" ","")
        .replace("\n","")
        .replace("True", "true")
        .replace("False", "false")
        .replace("None", "null")
        .replace("'",'"')
    )
    # Find all JSON objects in the text
    matches = list(re.finditer(r'\{.*?\}', text))
    if matches:
        # Get the last match
        last_match = matches[-1]
        try:
            json_data = json.loads(last_match.group())
            return json_data
        except json.JSONDecodeError as e:
            print(last_match.group())
            print("failed", e)
            return {}
    else:
        print("format failed")
        return {}

In [8]:
QWEN3_4B_DRITT_PATH= r"/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/analysis_data/llm_predictions/DATASET_ocr_MODEL_Qwen-4B_PROMPT_TARGET_drittauskunft_PROMPT_VERSION_V1_2026-06-01.csv"
QWEN35_4B_DRITT_PATH= r"/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/analysis_data/llm_predictions/DATASET_ocr_MODEL_Qwen-3.5-4B_PROMPT_TARGET_drittauskunft_PROMPT_VERSION_V1_2026-06-01.csv" 
QWEN3_32B_DRITT_PATH= r"/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/analysis_data/llm_predictions/DATASET_ocr_MODEL_Qwen-32B_PROMPT_TARGET_drittauskunft_PROMPT_VERSION_V1_2026-06-01.csv"


In [9]:
# read datasets
df_qwen3_4b_dritt = pd.read_csv(QWEN3_4B_DRITT_PATH)
df_qwen35_4b_dritt = pd.read_csv(QWEN35_4B_DRITT_PATH)
df_qwen3_32b_dritt = pd.read_csv(QWEN3_32B_DRITT_PATH)

In [10]:
df_qwen3_32b_dritt.document_type.value_counts()

document_type
attachment_and_transfer_order             150
ladung_va                                 150
approved_seizure                          150
court_inbox                               150
monierung_mb                              150
mail_attachments                          150
fp_protocol                               150
fp_invoice                                128
vermögensverzeichnis                       88
approved_attachment_and_transfer_order     64
drittauskunft                              53
enforcement_order                          23
tbd                                        22
bailiff_ip                                 20
contradiction                               3
neg_drittauskunft_hard                      1
Name: count, dtype: int64

In [11]:
df_qwen3_4b_dritt.pred_llm_drittauskunft.isna().sum()

np.int64(0)

In [12]:
df_qwen3_32b_dritt.pred_llm_drittauskunft.isna().sum()

np.int64(0)

In [13]:
df_qwen3_4b_dritt.ticket_uuid.is_unique

True

In [14]:
df_qwen3_4b_dritt.rename(columns={"pred_llm_drittauskunft":"pred_q3_4b_dritt"}, inplace=True)
df_qwen3_32b_dritt.rename(columns={"pred_llm_drittauskunft":"pred_q3_32b_dritt"}, inplace=True)
df_qwen35_4b_dritt.rename(columns={"pred_llm_drittauskunft":"pred_q35_4b_dritt"}, inplace=True)

In [15]:
pred_data = df_qwen3_4b_dritt.merge(df_qwen3_32b_dritt[["ticket_uuid", "pred_q3_32b_dritt"]], on="ticket_uuid", how="inner")
pred_data = pred_data.merge(df_qwen35_4b_dritt[["ticket_uuid", "pred_q35_4b_dritt"]], on="ticket_uuid", how="inner")

In [16]:
pred_data

,ticket_uuid,attachment_id,text,object_key,document_type,clean_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,text_w_pages,is_da_with_invoice,is_va,lentgh,pred_q3_4b_dritt,pred_q3_32b_dritt,pred_q35_4b_dritt
0,1ad77ddb-66ff-5624-b306-5e42e10d83b8,77842da0-8cc6-5fec-a50f-54c2932b2fb2,Antrag auf Erlass eines Pfändu...\nÜberweisung...,08.05.2023/1683556496_DS_001_08052023_161345.pdf,attachment_and_transfer_order,antrag auf erlass eines pfändu...\nüberweisung...,NaN,False,False,NaN,f4f127a70fcef56d3b74ad20682c59cc6e1d2d03923834...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,2603,"Okay, let's tackle this. The user wants to kno...","Okay, let's tackle this problem step by step. ...",The user wants me to classify a German legal d...
1,8865f2cc-a82a-5470-9601-66b749f1c680,b13cbfb6-fda9-53de-87e3-b1ef0a0d6330,Antrag auf Erlass eines Pfändu\nÜberweisungsbe...,09.02.2023/1675958832_Scan_HV_001_09022023_164...,attachment_and_transfer_order,antrag auf erlass eines pfändu\nüberweisungsbe...,NaN,False,False,NaN,16931a12da6b3b65a78d74303b6b635ed9216b5d67df05...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,12773,"Okay, let's tackle this. The user wants to kno...","Okay, let's tackle this problem step by step. ...",The user wants to classify a document as conta...
2,54306caa-b8e6-5ab1-b404-6696e0db4e43,e6238ebd-4e33-562a-939f-207f69c21116,Antrag auf Erlass eines unu\n1\nÜberweisungsbe...,11.05.2023/1683801369_Scan_AK_005_11052023_121...,attachment_and_transfer_order,antrag auf erlass eines unu\nüberweisungsbesch...,NaN,False,False,NaN,0be98a0b3e3cfe02500bcb8c5d09f5e820018d7126df51...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,2633,"Okay, let's tackle this. The user wants to kno...","Okay, let's tackle this. I need to determine i...",Thinking Process:\n\n1. **Analyze the Request...
3,8a2d6211-ef2b-5aed-b589-8422fb07251d,6b1d8f3c-e9ac-5d65-a9a0-a91834db6a60,Antrag auf Erlass eines Pfändungs- una\n1\nÜbe...,16.08.2023/1692180308_SK_001_16082023_114455.pdf,attachment_and_transfer_order,antrag auf erlass eines pfändungs- una\nüberwe...,NaN,False,False,NaN,705e1764544006710cb46cac06c5686215b42653e20a29...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,2679,"Okay, let's tackle this. The user wants to kno...","Okay, let's tackle this classification task. T...",Thinking Process:\n\n1. **Analyze the Request...
4,e2482fb5-4d67-5836-8c87-39d0e17e105c,f3982f7e-3ea7-5abd-9212-298da881dcdd,Antrag auf Erlass eines Pfäll\n1\nÜberweisungs...,05.12.2023/1701792298_MP_001_05122023_165102.pdf,attachment_and_transfer_order,antrag auf erlass eines pfäll\nüberweisungsbes...,NaN,False,False,NaN,515ca20bf62b661992eedca7fc3424f9895622850b7cad...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,2623,"Okay, let's tackle this. The user wants to kno...","Okay, let me tackle this problem step by step....",The user wants me to classify a document as co...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1447,421c6599-37a0-5822-8d21-11de81eadee9,1ccc264b-566c-535f-a5aa-d2ea89e8547f,Bundeszentralamt\nfür Steuern\nPOSTANSCHRIFT\...,data/aftercourt/drittauskunf_with_invoice/comb...,drittauskunft,bundeszentralamt\nfür steuern\npostanschrift\n...,NaN,False,False,NaN,481d84aab52bb306dd926fb8952770634cac437de5d388...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nBundeszentralamt\nfür Steuern\nPOSTA...,True,False,6951,"Okay, let's tackle this classification task. T...","Okay, let's tackle this. I need to determine i...",The user wants me to classify whether the inpu...
1448,d9f6ba15-63cf-5743-aa24-12f26140ace4,8b976e45-a27a-5f35-8f19-9c04a4a315bb,Neue Büroanschrift ab 01.12.2024\nDortmunder ...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,neue büroanschrift ab 01.12.2024\ndortmunder s...,NaN,False,False,NaN,1d6062807bfb02abacfe2291ee4b68bcd1568d401b1acd...,s3://pai

In [17]:
pred_data["pred_json_q35_4b_dritt"] = pred_data["pred_q35_4b_dritt"].apply(extract_last_json)

format failed
format failed
format failed
format failed
format failed
format failed
format failed
format failed
format failed
{"output":{"is_dritt":false}
failed Expecting ',' delimiter: line 1 column 29 (char 28)
format failed
format failed
format failed
format failed
format failed
format failed
format failed
format failed


In [18]:
pred_data["pred_json_q32b_dritt"] = pred_data["pred_q3_32b_dritt"].apply(extract_first_json)


In [ ]:
pred_data["pred_json_q4b_dritt"] = pred_data["pred_q3_4b_dritt"].apply(extract_first_json)

In [20]:
pred_data = pred_data[pred_data["pred_json_q35_4b_dritt"] != {}].copy()

In [21]:
pred_data.document_type.value_counts()

document_type
attachment_and_transfer_order             150
ladung_va                                 150
approved_seizure                          150
court_inbox                               150
monierung_mb                              150
mail_attachments                          150
fp_protocol                               143
fp_invoice                                123
vermögensverzeichnis                       84
approved_attachment_and_transfer_order     64
drittauskunft                              52
enforcement_order                          23
tbd                                        22
bailiff_ip                                 19
contradiction                               3
neg_drittauskunft_hard                      1
Name: count, dtype: int64

In [22]:
# extract is_erlass and is_invoice from pred_json
pred_data["q3_4b_is_dritt"] = pred_data["pred_json_q4b_dritt"].apply(lambda x: x.get("is_dritt", None))
pred_data["q3_32b_is_dritt"] = pred_data["pred_json_q32b_dritt"].apply(lambda x: x.get("is_dritt", None))
pred_data["q35_4b_is_dritt"] = pred_data["pred_json_q35_4b_dritt"].apply(lambda x: x.get("is_dritt", None))


In [23]:
pred_data

,ticket_uuid,attachment_id,text,object_key,document_type,clean_text,data,is_pfub,is_ladung,s3_link,...,lentgh,pred_q3_4b_dritt,pred_q3_32b_dritt,pred_q35_4b_dritt,pred_json_q35_4b_dritt,pred_json_q32b_dritt,pred_json_q4b_dritt,q3_4b_is_dritt,q3_32b_is_dritt,q35_4b_is_dritt
0,1ad77ddb-66ff-5624-b306-5e42e10d83b8,77842da0-8cc6-5fec-a50f-54c2932b2fb2,Antrag auf Erlass eines Pfändu...\nÜberweisung...,08.05.2023/1683556496_DS_001_08052023_161345.pdf,attachment_and_transfer_order,antrag auf erlass eines pfändu...\nüberweisung...,NaN,False,False,NaN,...,2603,"Okay, let's tackle this. The user wants to kno...","Okay, let's tackle this problem step by step. ...",The user wants me to classify a German legal d...,{'is_dritt': False},{'is_dritt': False},{'is_dritt': False},False,False,False
1,8865f2cc-a82a-5470-9601-66b749f1c680,b13cbfb6-fda9-53de-87e3-b1ef0a0d6330,Antrag auf Erlass eines Pfändu\nÜberweisungsbe...,09.02.2023/1675958832_Scan_HV_001_09022023_164...,attachment_and_transfer_order,antrag auf erlass eines pfändu\nüberweisungsbe...,NaN,False,False,NaN,...,12773,"Okay, let's tackle this. The user wants to kno...","Okay, let's tackle this problem step by step. ...",The user wants to classify a document as conta...,{'is_dritt': False},{'is_dritt': False},{'is_dritt': False},False,False,False
2,54306caa-b8e6-5ab1-b404-6696e0db4e43,e6238ebd-4e33-562a-939f-207f69c21116,Antrag auf Erlass eines unu\n1\nÜberweisungsbe...,11.05.2023/1683801369_Scan_AK_005_11052023_121...,attachment_and_transfer_order,antrag auf erlass eines unu\nüberweisungsbesch...,NaN,False,False,NaN,...,2633,"Okay, let's tackle this. The user wants to kno...","Okay, let's tackle this. I need to determine i...",Thinking Process:\n\n1. **Analyze the Request...,{'is_dritt': False},{'is_dritt': False},{'is_dritt': False},False,False,False
3,8a2d6211-ef2b-5aed-b589-8422fb07251d,6b1d8f3c-e9ac-5d65-a9a0-a91834db6a60,Antrag auf Erlass eines Pfändungs- una\n1\nÜbe...,16.08.2023/1692180308_SK_001_16082023_114455.pdf,attachment_and_transfer_order,antrag auf erlass eines pfändungs- una\nüberwe...,NaN,False,False,NaN,...,2679,"Okay, let's tackle this. The user wants to kno...","Okay, let's tackle this classification task. T...",Thinking Process:\n\n1. **Analyze the Request...,{'is_dritt': False},{'is_dritt': False},{'is_dritt': False},False,False,False
4,e2482fb5-4d67-5836-8c87-39d0e17e105c,f3982f7e-3ea7-5abd-9212-298da881dcdd,Antrag auf Erlass eines Pfäll\n1\nÜberweisungs...,05.12.2023/1701792298_MP_001_05122023_165102.pdf,attachment_and_transfer_order,antrag auf erlass eines pfäll\nüberweisungsbes...,NaN,False,False,NaN,...,2623,"Okay, let's tackle this. The user wants to kno...","Okay, let me tackle this problem step by step....",The user wants me to classify a document as co...,{'is_dritt': False},{'is_dritt': False},{'is_dritt': False},False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1447,421c6599-37a0-5822-8d21-11de81eadee9,1ccc264b-566c-535f-a5aa-d2ea89e8547f,Bundeszentralamt\nfür Steuern\nPOSTANSCHRIFT\...,data/aftercourt/drittauskunf_with_invoice/comb...,drittauskunft,bundeszentralamt\nfür steuern\npostanschrift\n...,NaN,False,False,NaN,...,6951,"Okay, let's tackle this classification task. T...","Okay, let's tackle this. I need to determine i...",The user wants me to classify whether the inpu...,{'is_dritt': True},{'is_dritt': True},{'is_dritt': True},True,True,True
1448,d9f6ba15-63cf-5743-aa24-12f26140ace4,8b976e45-a27a-5f35-8f19-9c04a4a315bb,Neue Büroanschrift ab 01.12.2024\nDortmunder ...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,neue büroanschrift ab 01.12.2024\ndortmunder s...,NaN,False,False,NaN,...,5329,"Okay, let's tackle this. The user wants to det...","Okay, let me tackle this classification proble...",The user wants me to classify the input text a...,{'is_dritt': True},{'is_dritt': True},{'is_dritt': True},True,True,True
1449,c72625b3-d66a-5df5-a933-11f57215838a,2a6e990a-3458-5efe-a8

In [24]:
total_different = (pred_data["q3_4b_is_dritt"] != pred_data["q3_32b_is_dritt"]).sum()
print(f"Total different predictions between Qwen 4b and 32b: {total_different} out of {len(pred_data)} samples")

Total different predictions between Qwen 4b and 32b: 53 out of 1434 samples


In [25]:
# q35-4b vs q32b
total_different = (pred_data["q35_4b_is_dritt"] != pred_data["q3_32b_is_dritt"]).sum()
print(f"Total different predictions between Qwen 3.5-4b and 32b: {total_different} out of {len(pred_data)} samples")

Total different predictions between Qwen 3.5-4b and 32b: 21 out of 1434 samples


In [26]:
# q35-4b vs q4b
total_different = (pred_data["q35_4b_is_dritt"] != pred_data["q3_4b_is_dritt"]).sum()
print(f"Total different predictions between Qwen 3.5-4b and 4b: {total_different} out of {len(pred_data)} samples")

Total different predictions between Qwen 3.5-4b and 4b: 50 out of 1434 samples


In [27]:
pred_data.document_type.value_counts()

document_type
attachment_and_transfer_order             150
ladung_va                                 150
approved_seizure                          150
court_inbox                               150
monierung_mb                              150
mail_attachments                          150
fp_protocol                               143
fp_invoice                                123
vermögensverzeichnis                       84
approved_attachment_and_transfer_order     64
drittauskunft                              52
enforcement_order                          23
tbd                                        22
bailiff_ip                                 19
contradiction                               3
neg_drittauskunft_hard                      1
Name: count, dtype: int64

In [28]:
gt = pred_data["document_type"].apply(lambda x: True if x == "drittauskunft" else False)
print(gt.value_counts())

document_type
False    1382
True       52
Name: count, dtype: int64


In [29]:
# calculate precision, recall, f1-score for each model
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

model_cols = {
    "Qwen3-4B": "q3_4b_is_dritt",
    "Qwen3-32B": "q3_32b_is_dritt",
    "Qwen3.5-4B": "q35_4b_is_dritt",
}

rows = []
for name, col in model_cols.items():
    pred = pred_data[col].fillna(False).astype(bool)
    rows.append({
        "model": name,
        "precision": precision_score(gt, pred, zero_division=0),
        "recall": recall_score(gt, pred, zero_division=0),
        "f1": f1_score(gt, pred, zero_division=0),
        "accuracy": accuracy_score(gt, pred),
    })

metrics_df = pd.DataFrame(rows).set_index("model")
metrics_df

,precision,recall,f1,accuracy
model,,,,
Qwen3-4B,0.288889,1.000000,0.448276,0.910739
Qwen3-32B,0.377778,0.980769,0.545455,0.940725
Qwen3.5-4B,0.375000,0.980769,0.542553,0.940028


##  Check false positives using qwen32b. There is some wrongly labelled examples probably

In [30]:
qwen32b_pred = pd.read_csv(QWEN3_32B_DRITT_PATH)
qwen32b_pred['q3_32b_is_dritt'] = qwen32b_pred['pred_llm_drittauskunft'].apply(lambda x: extract_first_json(x).get("is_dritt", None))
gt = qwen32b_pred['document_type'].apply(lambda x: True if x == "drittauskunft" else False)
print(qwen32b_pred['q3_32b_is_dritt'].value_counts())

precision_score(gt, qwen32b_pred['q3_32b_is_dritt'].fillna(False).astype(bool), zero_division=0)


q3_32b_is_dritt
False    1312
True      140
Name: count, dtype: int64


0.37142857142857144

In [31]:
fps = qwen32b_pred[(gt == False) & (qwen32b_pred['q3_32b_is_dritt'] == True)]
fps

,ticket_uuid,attachment_id,text,object_key,document_type,clean_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,text_w_pages,is_da_with_invoice,is_va,lentgh,pred_llm_drittauskunft,q3_32b_is_dritt
542,ca025357-d3f5-5b97-a780-a4725e3d6239,6cd033bf-c19e-53d4-af67-c5157c19991e,Svea Rietschek\nVolksparkstraße 52\nObergerich...,NaN,fp_invoice,svea rietschek\nvolksparkstraße 52\nobergerich...,NaN,False,False,NaN,6c3cea7cd3eb08d8d6f92e35bb0e258a9f3f35d271df94...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,7895,"Okay, let's tackle this. I need to determine i...",True
544,61cc50ed-0907-5128-ac81-436af141eff6,33b683fa-ab39-508f-887f-4d7c5bc28da1,Marstallstr. 15\nGerichtsvollzieherin\n68723 S...,NaN,fp_invoice,marstallstr. 15\ngerichtsvollzieherin\n68723 s...,NaN,False,False,NaN,ae0a4493bec01105036a5415698f9bdd8e12ebbce23d7a...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,2623,"Okay, let's tackle this. I need to determine i...",True
548,7f52f4f1-8921-528d-aba3-034c48b90be8,087b8cac-4ccb-54cc-af40-dbf1b087ac32,Beethovenstr. 22a\nGERIT FRIES\n16259 Bad Frei...,NaN,fp_invoice,beethovenstr. 22a\ngerit fries\n16259 bad frei...,NaN,False,False,NaN,c16246813f92f8040c23d84f96ab74b436b3bb2c5b2576...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,2646,"Okay, let's tackle this classification task. T...",True
550,68721968-961a-505b-9950-34b45e95886d,e3144789-511c-5315-b53c-2b3b4cae9cb2,Beethovenstr. 22a\nGERIT FRIES\n16259 Bad Frei...,NaN,fp_invoice,beethovenstr. 22a\ngerit fries\n16259 bad frei...,NaN,False,False,NaN,c16246813f92f8040c23d84f96ab74b436b3bb2c5b2576...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,2771,"Okay, let's tackle this. I need to determine i...",True
552,359f5283-f62e-5ac6-95a9-bd5b9cab3aaf,618f325d-5cbd-5784-8df2-afc63d55c4b7,Domkauler Weg 58\n50171 Kerpen\nTel. 02275/911...,11.10.2024/ocr-v2_KL_003-11102024-115311.pdf,fp_invoice,domkauler weg 58\n50171 kerpen\ntel. 02275/911...,NaN,False,False,NaN,a8f0910dca00d02d632e90821f68af3c27530b7407a2aa...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,3292,"Okay, let's tackle this. I need to determine i...",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1357,48c9b11b-b994-5b0d-bd47-760fc1a6c8ab,49548159,DANA PRZYBILLA\nDie nachstehend gewählten Form...,NaN,fp_protocol,dana przybilla\ndie nachstehend gewählten form...,NaN,False,False,s3://pair-data-engineering-new/ocr_prepared_ou...,7d7e239a81e62ae7d314f31199533327772b415d413aa9...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,4304,"Okay, let's tackle this. I need to determine i...",True
1363,e2efebeb-fc83-5862-afe8-9570236fe309,47477600,PETRA MANN\nDie nachstehend gewählten Formulie...,NaN,fp_protocol,petra mann\ndie nachstehend gewählten formulie...,NaN,False,False,s3://pair-data-engineering-new/ocr_prepared_ou...,b5557c61dcd15ea259be6e7c0f67a64e8711c57046e444...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,4430,"Okay, let's tackle this problem step by step. ...",True
1370,2fabba5f-56f4-5ac1-9ac4-de222ac9f1ff,45189489,FRANK NOWAK\nDie nachstehend gewählten Formuli...,NaN,fp_protocol,frank nowak\ndie nachstehend gewählten formuli...,NaN,False,False,s3://pair-data-engineering-new/ocr_prepared_ou...,c94bf0a4d483c1e2fe9bef9eb004690b0a1c11b75ee294...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,4467,"Okay, let's tackle this classification task. T...",True
1376,ac20d00a-ffaa-5a7b-8752-bd855ed494f9,47481327,PETRA MANN\nDie nachstehend gewählten Formulie...,NaN,fp_protocol,petra mann\ndie nachstehend gewählten formulie...,NaN,False,False,s3://pair-data-engineering-new/ocr_prepared_ou...,b5557c61dcd15ea259be6e7c0f67a64e8711c57046e444...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,4411,"Okay, let's tackle this. I need to determine i...",True


In [32]:
fps_with_object_key = fps[fps['object_key'].notna()]
fps_with_object_key = fps_with_object_key.reset_index(drop=True)
fps_with_object_key

,ticket_uuid,attachment_id,text,object_key,document_type,clean_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,text_w_pages,is_da_with_invoice,is_va,lentgh,pred_llm_drittauskunft,q3_32b_is_dritt
0,359f5283-f62e-5ac6-95a9-bd5b9cab3aaf,618f325d-5cbd-5784-8df2-afc63d55c4b7,Domkauler Weg 58\n50171 Kerpen\nTel. 02275/911...,11.10.2024/ocr-v2_KL_003-11102024-115311.pdf,fp_invoice,domkauler weg 58\n50171 kerpen\ntel. 02275/911...,NaN,False,False,NaN,a8f0910dca00d02d632e90821f68af3c27530b7407a2aa...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,3292,"Okay, let's tackle this. I need to determine i...",True
1,b752f579-e1f8-50a5-91a8-1fd0f7b6fe30,f7efdaeb-7689-5c40-a276-741cde398b60,Donauwörther Str. 169\nNADINE GÄRTNER\n86154 A...,18.10.2024/ocr-v2_DR-II_168524_Nachr_4_15_Erte...,fp_invoice,donauwörther str. 169\nnadine gärtner\n86154 a...,NaN,False,False,NaN,b61bf5517241896877769385448766db92bd7c957e5a77...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,3000,"Okay, let's tackle this classification step by...",True
2,91b18d43-4ebc-52f5-ae81-bbb3ec4404db,19645685-662d-5918-a956-6a08d3eb39e6,Lindenstr. 19\nMICHAEL SCHWENKE\n06749 Bitterf...,31.07.2024/ocr-v2_DR-II_076724_Nachr_ev18_30_0...,fp_invoice,lindenstr. 19\nmichael schwenke\n06749 bitterf...,NaN,False,False,NaN,4943c407af54d74a814b80200c757340b7deca5a273411...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,2182,"Okay, let's tackle this. I need to determine i...",True
3,110bbd40-601f-57bc-a3d6-8854594adc8d,a605eba7-c7b6-59dc-925a-1199a1455e75,Gerichtsvollzieherin\nAmtsgericht\nEttlingen\n...,31.07.2024/ocr-v2_DR_II_719_24_Gl_Haft_Vermöge...,fp_invoice,gerichtsvollzieherin\namtsgericht\nettlingen\n...,NaN,False,False,NaN,9839cd7c1be7c000619b4344a3f3265408c3991298cf90...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,7756,"Okay, let me tackle this step by step. I need ...",True
4,bdab721e-a8e4-52b7-b8d6-d36026bde6d5,036890bd-22ea-591c-ac8b-8b2daa82caae,Obergerichtsvollzieher\nLammstr. 11\nK. Nöbaue...,24.09.2024/ocr-v2_GVZ_2_dr_1002_24_an_gl_nachr...,fp_invoice,obergerichtsvollzieher\nlammstr. 11\nk. nöbaue...,NaN,False,False,NaN,b404373ec0f48595e93c744e152a4840529c40fe946264...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,31156,"Okay, let's tackle this problem step by step. ...",True
5,c518854d-2f65-53af-a4b3-d8b62df46502,e4b3c249-4de5-5cee-8e8f-32bd8ad0b5d9,"Gabriele Kruse\nKönigstraße 11, Zi EG 005\nObe...",11.12.2024/ocr-v2_Dokumente_45924_10122024_112...,fp_invoice,"gabriele kruse\nkönigstraße 11, zi eg 005\nobe...",NaN,False,False,NaN,292d99228bc98d5911d52d718ad555c152b3833d968164...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,5652,"Okay, let me try to figure this out. I need to...",True
6,416f3155-cfc2-5111-9747-7f9852c176ad,051fc4de-d819-5e51-8fae-e72898717f6e,Hubertus Kohlrautz\nAlfred-Delp-Weg 2\nOberger...,02.12.2024/ocr-v2_dokument_102924_13112024_110...,fp_invoice,hubertus kohlrautz\nalfred-delp-weg 2\noberger...,NaN,False,False,NaN,4ca7e3f068d709aa637c4b868ccf0d51ee21b3400e84af...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,4390,"Okay, let's tackle this document classificatio...",True
7,ec8d9899-9fa9-58cb-9649-8a912e3fe1de,fd87d520-cbb1-591b-8c7a-3ea2347d4f95,Olivia Günther\nObergerichtsvollzieherin\nAmts...,31.07.2024/ocr-v2_1_sammel1.pdf,fp_invoice,olivia günther\nobergerichtsvollzieherin\namts...,NaN,False,False,NaN,eabf55f9405fd79f4e54e5c5168398f08169a432152b70...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,18948,"Okay, let me try to work through this step by ...",True
8,d006bb6e-91d4-591c-a83a-5d42cea38f8f,ac6d1964-9c0c-58e0-b83f-64bb709ca1bb,Franziska Kluge\nOsterbrooksweg 42+44\nJHS'in ...,20.06.2024/ocr-v2_Dokumente_4624_20062024_0851...,fp_invoice,franziska kluge\nosterbrooksweg 42+44\njhs'in ...,NaN,False,False,NaN,5e6ec9ead74f6dc6f85049ea399eb

#### For ocr dataset files, read the data from pair-scanner/['object_key']. It is directly under pair-scanner

In [28]:
import sys
import os
sys.path.append("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation")

from utils.prod_utils import download_pdf_by_document_s3_info
import boto3

session = boto3.Session(profile_name="739275445236_DataScienceUser")
s3 = session.client("s3")
download_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/dritt_fps"


In [37]:
def download_files_in_sample(sample_df, download_dir):
    os.makedirs(download_dir, exist_ok=True)
    idx = 0
    for _, row in sample_df.iterrows():
        
        print(f"Downloading file for attachment_id: {row['attachment_id']}, idx: {idx}")
        download_pdf_by_document_s3_info(
            s3=s3,
            document_s3_bucket=row["document_s3_bucket"],
            document_s3_key=row["document_s3_key"],
            download_dir=download_dir,
            file_name=row.get("file_name", None)
        )
        idx += 1 
        
def download_files_in_sample_OCR_dataset(sample_df, download_dir):
    os.makedirs(download_dir, exist_ok=True)
    idx = 0
    download_local_dir = []
    for _, row in sample_df.iterrows():
        
        print(f"Downloading file for attachment_id: {row['attachment_id']}, idx: {idx}")
        try:
            download_pdf_by_document_s3_info(
                s3=s3,
                document_s3_bucket="pair-scanner",
                document_s3_key=row["object_key"],
                download_dir=download_dir,
                file_name=row.get("file_name", None)
            )
            download_local_dir.append(os.path.join(download_dir, row.get("file_name", None) if row.get("file_name", None) else os.path.basename(row["object_key"])))
        except Exception as e:
            download_local_dir.append(None) # to keep the order, append None if download failed
            print(f"Failed to download file for attachment_id: {row['attachment_id']}, idx: {idx}. Error: {e}")
        idx += 1
    return download_local_dir

In [38]:
# sample = false_positives.sample(n=100, random_state=42)
sample = fps_with_object_key.copy() # take all
local_dirs = download_files_in_sample_OCR_dataset(sample, download_dir)

✅ PDF downloaded successfully: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/dritt_fps/ocr-v2_KL_003-11102024-115311.pdf
✅ PDF downloaded successfully: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/dritt_fps/ocr-v2_DR-II_168524_Nachr_4_15_Erteilung_Vermoegensverzeichnis_an_Folgeglaeubiger_Ansch_17_10.pdf
✅ PDF downloaded successfully: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/dritt_fps/ocr-v2_DR-II_076724_Nachr_ev18_30_07_2024.pdf
✅ PDF downloaded successfully: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/dritt_fps/ocr-v2_DR_II_719_24_Gl_Haft_Vermögensauskunft.PDF.pdf
✅ PDF downloaded successfully: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/dritt_fps/ocr-v2_GVZ_2_dr_1002_24_an_gl_nachricht_hbantrag.pdf
✅ PDF downloaded successfully: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/dritt_fps/ocr-v2_Dokume

In [39]:
sample['local_dir'] = local_dirs

In [40]:
sample.to_csv('streamlit_use_dritt_fp.csv', index=False)

In [41]:
sample

,ticket_uuid,attachment_id,text,object_key,document_type,clean_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,text_w_pages,is_da_with_invoice,is_va,lentgh,pred_llm_drittauskunft,q3_32b_is_dritt,local_dir
0,359f5283-f62e-5ac6-95a9-bd5b9cab3aaf,618f325d-5cbd-5784-8df2-afc63d55c4b7,Domkauler Weg 58\n50171 Kerpen\nTel. 02275/911...,11.10.2024/ocr-v2_KL_003-11102024-115311.pdf,fp_invoice,domkauler weg 58\n50171 kerpen\ntel. 02275/911...,NaN,False,False,NaN,a8f0910dca00d02d632e90821f68af3c27530b7407a2aa...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,3292,"Okay, let's tackle this. I need to determine i...",True,/Users/melih.gorgulu/Desktop/Projects/aftercou...
1,b752f579-e1f8-50a5-91a8-1fd0f7b6fe30,f7efdaeb-7689-5c40-a276-741cde398b60,Donauwörther Str. 169\nNADINE GÄRTNER\n86154 A...,18.10.2024/ocr-v2_DR-II_168524_Nachr_4_15_Erte...,fp_invoice,donauwörther str. 169\nnadine gärtner\n86154 a...,NaN,False,False,NaN,b61bf5517241896877769385448766db92bd7c957e5a77...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,3000,"Okay, let's tackle this classification step by...",True,/Users/melih.gorgulu/Desktop/Projects/aftercou...
2,91b18d43-4ebc-52f5-ae81-bbb3ec4404db,19645685-662d-5918-a956-6a08d3eb39e6,Lindenstr. 19\nMICHAEL SCHWENKE\n06749 Bitterf...,31.07.2024/ocr-v2_DR-II_076724_Nachr_ev18_30_0...,fp_invoice,lindenstr. 19\nmichael schwenke\n06749 bitterf...,NaN,False,False,NaN,4943c407af54d74a814b80200c757340b7deca5a273411...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,2182,"Okay, let's tackle this. I need to determine i...",True,/Users/melih.gorgulu/Desktop/Projects/aftercou...
3,110bbd40-601f-57bc-a3d6-8854594adc8d,a605eba7-c7b6-59dc-925a-1199a1455e75,Gerichtsvollzieherin\nAmtsgericht\nEttlingen\n...,31.07.2024/ocr-v2_DR_II_719_24_Gl_Haft_Vermöge...,fp_invoice,gerichtsvollzieherin\namtsgericht\nettlingen\n...,NaN,False,False,NaN,9839cd7c1be7c000619b4344a3f3265408c3991298cf90...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,7756,"Okay, let me tackle this step by step. I need ...",True,/Users/melih.gorgulu/Desktop/Projects/aftercou...
4,bdab721e-a8e4-52b7-b8d6-d36026bde6d5,036890bd-22ea-591c-ac8b-8b2daa82caae,Obergerichtsvollzieher\nLammstr. 11\nK. Nöbaue...,24.09.2024/ocr-v2_GVZ_2_dr_1002_24_an_gl_nachr...,fp_invoice,obergerichtsvollzieher\nlammstr. 11\nk. nöbaue...,NaN,False,False,NaN,b404373ec0f48595e93c744e152a4840529c40fe946264...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,31156,"Okay, let's tackle this problem step by step. ...",True,/Users/melih.gorgulu/Desktop/Projects/aftercou...
5,c518854d-2f65-53af-a4b3-d8b62df46502,e4b3c249-4de5-5cee-8e8f-32bd8ad0b5d9,"Gabriele Kruse\nKönigstraße 11, Zi EG 005\nObe...",11.12.2024/ocr-v2_Dokumente_45924_10122024_112...,fp_invoice,"gabriele kruse\nkönigstraße 11, zi eg 005\nobe...",NaN,False,False,NaN,292d99228bc98d5911d52d718ad555c152b3833d968164...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,5652,"Okay, let me try to figure this out. I need to...",True,/Users/melih.gorgulu/Desktop/Projects/aftercou...
6,416f3155-cfc2-5111-9747-7f9852c176ad,051fc4de-d819-5e51-8fae-e72898717f6e,Hubertus Kohlrautz\nAlfred-Delp-Weg 2\nOberger...,02.12.2024/ocr-v2_dokument_102924_13112024_110...,fp_invoice,hubertus kohlrautz\nalfred-delp-weg 2\noberger...,NaN,False,False,NaN,4ca7e3f068d709aa637c4b868ccf0d51ee21b3400e84af...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,4390,"Okay, let's tackle this document classificatio...",True,/Users/melih.gorgulu/Desktop/Projects/aftercou...
7,ec8d9899-9fa9-58cb-9649-8a912e3fe1de,fd87d520-cbb1-591b-8c7a-3ea2347d4f95,Olivia Günther\nObergerichtsvollzieherin\nAmts...,31.07.2024/ocr-v2_1_sammel1.pdf,fp_invoice,olivia günther\nobergerichtsvollzieherin\namts...,NaN,False,False,NaN,eabf55f9405fd79f4e54e5c5168398f08169a432152b70...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN

In [ ]:
# e4b3c249-4de5-5cee-8e8f-32bd8ad0b5d9 -> bu normal fp OLMAYAN DRITT
# digerleri combination 

# THESE ARE ACTUALLY NOT FP:

# 036890bd-22ea-591c-ac8b-8b2daa82caae
# fd87d520-cbb1-591b-8c7a-3ea2347d4f95
# ac6d1964-9c0c-58e0-b83f-64bb709ca1bb
# bb182f24-e497-5382-9a8c-324419ab784d
# 03c4d3ec-d2a7-54d0-a416-d32434fafce5
# d360abbe-dff1-53aa-96e4-b9cf73ec9026
# 29c564aa-d8a3-5b26-8e20-03024909279d
# d976c5f6-e8ca-53a3-92f7-30e3a7fbad38
# 0f70a6e3-889f-5e39-9e37-27b2635f9bd1
# fddb5017-9036-5a73-a78d-a2b1bc7a75ce
# e4b3c249-4de5-5cee-8e8f-32bd8ad0b5d9 # NORMAL NOT FP. Rest is combination (protokol+dritt+invoice+ve)
# bc0d88ab-52cd-5f3a-99fc-c4eb1fe5e467
# 542972ee-feb8-5466-a7d3-11c4320e204b
# 9ee5bc1d-b55f-5250-98b7-2ab4bcf8bff0
# d1fcf449-6008-582c-ae09-fc1fa537d08b
# 08b99412-8ec4-5d2e-a4d2-9cdba232f019
# d5fa31bc-61e0-56f4-8988-8aa11828ba9b
# cb70343f-09f3-5927-9c0e-483283721511
# a605eba7-c7b6-59dc-925a-1199a1455e75

In [33]:
fps_with_no_object_key = fps[~fps['object_key'].notna()]
fps_with_no_object_key = fps_with_no_object_key.reset_index(drop=True)
fps_with_no_object_key

,ticket_uuid,attachment_id,text,object_key,document_type,clean_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,text_w_pages,is_da_with_invoice,is_va,lentgh,pred_llm_drittauskunft,q3_32b_is_dritt
0,ca025357-d3f5-5b97-a780-a4725e3d6239,6cd033bf-c19e-53d4-af67-c5157c19991e,Svea Rietschek\nVolksparkstraße 52\nObergerich...,NaN,fp_invoice,svea rietschek\nvolksparkstraße 52\nobergerich...,NaN,False,False,NaN,6c3cea7cd3eb08d8d6f92e35bb0e258a9f3f35d271df94...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,7895,"Okay, let's tackle this. I need to determine i...",True
1,61cc50ed-0907-5128-ac81-436af141eff6,33b683fa-ab39-508f-887f-4d7c5bc28da1,Marstallstr. 15\nGerichtsvollzieherin\n68723 S...,NaN,fp_invoice,marstallstr. 15\ngerichtsvollzieherin\n68723 s...,NaN,False,False,NaN,ae0a4493bec01105036a5415698f9bdd8e12ebbce23d7a...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,2623,"Okay, let's tackle this. I need to determine i...",True
2,7f52f4f1-8921-528d-aba3-034c48b90be8,087b8cac-4ccb-54cc-af40-dbf1b087ac32,Beethovenstr. 22a\nGERIT FRIES\n16259 Bad Frei...,NaN,fp_invoice,beethovenstr. 22a\ngerit fries\n16259 bad frei...,NaN,False,False,NaN,c16246813f92f8040c23d84f96ab74b436b3bb2c5b2576...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,2646,"Okay, let's tackle this classification task. T...",True
3,68721968-961a-505b-9950-34b45e95886d,e3144789-511c-5315-b53c-2b3b4cae9cb2,Beethovenstr. 22a\nGERIT FRIES\n16259 Bad Frei...,NaN,fp_invoice,beethovenstr. 22a\ngerit fries\n16259 bad frei...,NaN,False,False,NaN,c16246813f92f8040c23d84f96ab74b436b3bb2c5b2576...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,2771,"Okay, let's tackle this. I need to determine i...",True
4,c8860a21-bc31-5dcb-adb2-ef36b8bfa451,502c72b1-7571-5618-aafd-c2399aeebbbf,Christiane Soll\nBergstraße 5 7\nObergerichtsv...,NaN,fp_invoice,christiane soll\nbergstraße 5 7\nobergerichtsv...,NaN,False,False,NaN,87f4403803334c11786aad484948915ff3bd715347ca88...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,3008,"Okay, let's tackle this step by step. I need t...",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,48c9b11b-b994-5b0d-bd47-760fc1a6c8ab,49548159,DANA PRZYBILLA\nDie nachstehend gewählten Form...,NaN,fp_protocol,dana przybilla\ndie nachstehend gewählten form...,NaN,False,False,s3://pair-data-engineering-new/ocr_prepared_ou...,7d7e239a81e62ae7d314f31199533327772b415d413aa9...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,4304,"Okay, let's tackle this. I need to determine i...",True
57,e2efebeb-fc83-5862-afe8-9570236fe309,47477600,PETRA MANN\nDie nachstehend gewählten Formulie...,NaN,fp_protocol,petra mann\ndie nachstehend gewählten formulie...,NaN,False,False,s3://pair-data-engineering-new/ocr_prepared_ou...,b5557c61dcd15ea259be6e7c0f67a64e8711c57046e444...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,4430,"Okay, let's tackle this problem step by step. ...",True
58,2fabba5f-56f4-5ac1-9ac4-de222ac9f1ff,45189489,FRANK NOWAK\nDie nachstehend gewählten Formuli...,NaN,fp_protocol,frank nowak\ndie nachstehend gewählten formuli...,NaN,False,False,s3://pair-data-engineering-new/ocr_prepared_ou...,c94bf0a4d483c1e2fe9bef9eb004690b0a1c11b75ee294...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,4467,"Okay, let's tackle this classification task. T...",True
59,ac20d00a-ffaa-5a7b-8752-bd855ed494f9,47481327,PETRA MANN\nDie nachstehend gewählten Formulie...,NaN,fp_protocol,petra mann\ndie nachstehend gewählten formulie...,NaN,False,False,s3://pair-data-engineering-new/ocr_prepared_ou...,b5557c61dcd15ea259be6e7c0f67a64e8711c57046e444...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False,4411,"Okay, let's tackle this. I need to determine i...",True


In [34]:
import re
import html as _html
from IPython.display import HTML, display

# Keyword groups -> color. Patterns are tried in order; put more specific first.
HIGHLIGHT_GROUPS = {
    "drittauskunft": {
        "color": "#2e7d32",  # green
        "patterns": [
            r"Kontenabrufersuchen\s+nach\s+§§\s*93,\s*93b\s+Abgabenordnung\s*\(AO\)",
            r"Bundeszentralamt\s+für\s+Steuern(?:\s*/\s*BSZT)?",
            r"Anlage\s+zu\s+GZ:\s*St\s*II\s*4\s*-\s*S\s*0229a\s*-\s*KEVIZZ[^\n]*",
            r"St\s*II\s*4\s*-\s*S\s*0229a\s*-\s*KEVIZZ",
            r"Der\s+Abruf\s+hat\s+zu\s+keinem\s+Ergebnis\s+geführt\.?",
            r"mit\s+folgenden\s+Anhaltsdaten",
            r"zu\s+der\s+Anfrage\s+von\s+dem/der",
            r"Geburtsdatum",
            r"Nachname",
            r"Vorname",
            r"Ergebnisse",
            r"Ergebnis",
            r"BSZT",
            r"Name",
        ],
    },
    "vermoegensverzeichnis": {
        "color": "#c62828",  # red
        "patterns": [
            r"Vermögensverzeichnis\s+gemäß\s+§\s*802c\s+ZPO",
            r"Vermögensverzeichnis\s+des\s+Schuldners",
            r"Vermögensverzeichnis\s+der\s+Schuldnerin",
            r"Seite\s+\d+\s+von\s+\d+\s+des\s+Vermögensverzeichnisses",
            r"Vermögensverzeichnis",
        ],
    },
    "protokoll": {
        "color": "#1565c0",  # blue
        "patterns": [
            r"Protokoll\s+zur\s+Vermögensauskunft",
            r"Vermoegensauskunft",
            r"Vermögensauskunft[c]?",
            r"Protokoll",
        ],
    },
}


def highlight_ocr_text(text: str, as_html: bool = True):
    """Wrap matched keywords with <...> markers and color (HTML).

    The original matched text is preserved inside the angle brackets.
    Example: 'Protokoll' -> '<Protokoll>' rendered in blue.
    """
    parts = []
    color_for_index = []
    for cfg in HIGHLIGHT_GROUPS.values():
        for pat in cfg["patterns"]:
            parts.append(f"({pat})")
            color_for_index.append(cfg["color"])

    combined = re.compile("|".join(parts), flags=re.IGNORECASE)

    def _replace_plain(m: re.Match) -> str:
        return f"<{m.group(0)}>"

    if not as_html:
        return combined.sub(_replace_plain, text)

    # HTML mode: escape full text, then re-run regex on escaped text
    # (patterns only contain ASCII punctuation that survives escaping unchanged
    # except for spaces/letters, which are unaffected).
    escaped = _html.escape(text)

    def _replace_html(m: re.Match) -> str:
        idx = next(i for i, g in enumerate(m.groups()) if g is not None)
        color = color_for_index[idx]
        inner = m.group(0)
        return (
            f'<span style="color:{color};font-weight:600;'
            f'background:rgba(0,0,0,0.04);padding:1px 4px;border-radius:3px;">'
            f"&lt;{inner}&gt;</span>"
        )

    result = combined.sub(_replace_html, escaped).replace("\n", "<br>")
    return (
        '<div style="font-family:ui-monospace,Menlo,monospace;'
        'white-space:pre-wrap;line-height:1.5;font-size:13px;">'
        f"{result}</div>"
    )


def show_highlighted(text: str):
    display(HTML(highlight_ocr_text(text, as_html=True)))



In [35]:
fps_with_no_object_key["clean_text_colored"] = fps_with_no_object_key["clean_text"].apply(
    lambda t: highlight_ocr_text(t, as_html=True) if isinstance(t, str) else t
)


In [36]:
fps_with_no_object_key.to_csv('streamlit_use_dritt_fp_no_object_key.csv', index=False)

In [78]:
# nereler degismeli:
"""
1) Original raw text
2) llm input sample
3) predictions results for qwen32, qwen4b, qwen35-4b
"""


'\n1) Original raw text\n2) llm input sample\n3) predictions results for qwen32, qwen4b, qwen35-4b\n'

In [79]:
raw_data_path = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/data/raw/final_raw_data.csv"
llm_input_sample_path = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/analysis_data/llm_input_sample_dritt.csv"
pred_qwen32b_path = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/analysis_data/llm_predictions/DATASET_ocr_MODEL_Qwen-32B_PROMPT_TARGET_drittauskunft_PROMPT_VERSION_V1_2026-06-01.csv"
pred_qwen4b_path = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/analysis_data/llm_predictions/DATASET_ocr_MODEL_Qwen-4B_PROMPT_TARGET_drittauskunft_PROMPT_VERSION_V1_2026-06-01.csv"
pred_qwen35_4b_path = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/analysis_data/llm_predictions/DATASET_ocr_MODEL_Qwen-3.5-4B_PROMPT_TARGET_drittauskunft_PROMPT_VERSION_V1_2026-06-01.csv"

In [80]:
not_fp_attachment_ids_path_nopdf = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/llms/not_fp_attachment_ids_no_pdf.txt"
not_fp_attachment_ids_path = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/llms/not_fp_attachment_ids.txt"

In [81]:
# read 
with open(not_fp_attachment_ids_path, "r") as f:
    not_fp_attachment_ids = list(set(line.strip() for line in f if line.strip()))
with open(not_fp_attachment_ids_path_nopdf, "r") as f:
    not_fp_attachment_ids_no_pdf = list(set(line.strip() for line in f if line.strip()))

In [82]:
len(not_fp_attachment_ids)

19

In [83]:
len(not_fp_attachment_ids_no_pdf)

17

In [84]:
# e4b3c249-4de5-5cee-8e8f-32bd8ad0b5d9 -> this should be labelled as normal dritt, not combination
not_fp_attachment_ids.remove("e4b3c249-4de5-5cee-8e8f-32bd8ad0b5d9")
not_fp_attachment_ids_no_pdf.append("e4b3c249-4de5-5cee-8e8f-32bd8ad0b5d9")

In [85]:
len(not_fp_attachment_ids), len(not_fp_attachment_ids_no_pdf)

(18, 18)

In [86]:
# update raw data
raw_data = pd.read_csv(raw_data_path)
raw_data.columns

Index(['ticket_uuid', 'attachment_id', 'text', 'object_key', 'document_type',
       'cleaned_text', 'data', 'is_pfub', 'is_ladung', 's3_link',
       'textract_job_id', 'textract_s3_link', 'is_ve_with_invoice',
       'text_w_pages', 'is_da_with_invoice', 'is_va'],
      dtype='object')

In [87]:

# 1) update not_fp_attachment_ids -> their doc types is "va_dritt_invoice_protokol_combination"
raw_data.loc[raw_data["attachment_id"].isin(not_fp_attachment_ids), "document_type"] = "va_dritt_invoice_protokol_combination"
raw_data.loc[raw_data["attachment_id"].isin(not_fp_attachment_ids), "is_ve_with_invoice"] = True
raw_data.loc[raw_data["attachment_id"].isin(not_fp_attachment_ids), "is_da_with_invoice"] = True
raw_data.loc[raw_data["attachment_id"].isin(not_fp_attachment_ids), "is_va"] = True

# 2) update not_fp_attachment_ids_no_pdf -> their doc types is "drittauskunft" (normal dritt, no combination)
raw_data.loc[raw_data["attachment_id"].isin(not_fp_attachment_ids_no_pdf), "document_type"] = "drittauskunft"
raw_data.loc[raw_data["attachment_id"].isin(not_fp_attachment_ids_no_pdf), "is_ve_with_invoice"] = False

In [88]:
# Validate changes: compare before vs after (NaN-safe)
raw_data_orig = pd.read_csv(raw_data_path)

diff_mask = (raw_data != raw_data_orig) & ~(raw_data.isna() & raw_data_orig.isna())
changed_mask = diff_mask.any(axis=1)
print(f"Total rows changed: {changed_mask.sum()}")

changed_cols = diff_mask.columns[diff_mask.any(axis=0)].tolist()
print(f"\nColumns changed: {changed_cols}")
print(f"\nPer-column change counts:")
for col in changed_cols:
    print(f"  {col}: {diff_mask[col].sum()} rows")

# Breakdown by group
mask1 = raw_data["attachment_id"].isin(not_fp_attachment_ids)
mask2 = raw_data["attachment_id"].isin(not_fp_attachment_ids_no_pdf)
print(f"\nGroup 1 (combination): {mask1.sum()} rows matched out of {len(not_fp_attachment_ids)} ids")
print(f"Group 2 (drittauskunft): {mask2.sum()} rows matched out of {len(not_fp_attachment_ids_no_pdf)} ids")


Total rows changed: 36

Columns changed: ['document_type', 'is_ve_with_invoice', 'is_da_with_invoice', 'is_va']

Per-column change counts:
  document_type: 36 rows
  is_ve_with_invoice: 36 rows
  is_da_with_invoice: 18 rows
  is_va: 4 rows

Group 1 (combination): 18 rows matched out of 18 ids
Group 2 (drittauskunft): 18 rows matched out of 18 ids


In [89]:
# change llm_input_sample_path

llm_input_sample = pd.read_csv(llm_input_sample_path)
llm_input_sample

,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,text_w_pages,is_da_with_invoice,is_va
0,1ad77ddb-66ff-5624-b306-5e42e10d83b8,77842da0-8cc6-5fec-a50f-54c2932b2fb2,Antrag auf Erlass eines Pfändu...\nÜberweisung...,08.05.2023/1683556496_DS_001_08052023_161345.pdf,attachment_and_transfer_order,antrag auf erlass eines pfändu...\nüberweisung...,NaN,False,False,NaN,f4f127a70fcef56d3b74ad20682c59cc6e1d2d03923834...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
1,8865f2cc-a82a-5470-9601-66b749f1c680,b13cbfb6-fda9-53de-87e3-b1ef0a0d6330,Antrag auf Erlass eines Pfändu\nÜberweisungsbe...,09.02.2023/1675958832_Scan_HV_001_09022023_164...,attachment_and_transfer_order,antrag auf erlass eines pfändu\nüberweisungsbe...,NaN,False,False,NaN,16931a12da6b3b65a78d74303b6b635ed9216b5d67df05...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
2,54306caa-b8e6-5ab1-b404-6696e0db4e43,e6238ebd-4e33-562a-939f-207f69c21116,Antrag auf Erlass eines unu\n1\nÜberweisungsbe...,11.05.2023/1683801369_Scan_AK_005_11052023_121...,attachment_and_transfer_order,antrag auf erlass eines unu\nüberweisungsbesch...,NaN,False,False,NaN,0be98a0b3e3cfe02500bcb8c5d09f5e820018d7126df51...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
3,8a2d6211-ef2b-5aed-b589-8422fb07251d,6b1d8f3c-e9ac-5d65-a9a0-a91834db6a60,Antrag auf Erlass eines Pfändungs- una\n1\nÜbe...,16.08.2023/1692180308_SK_001_16082023_114455.pdf,attachment_and_transfer_order,antrag auf erlass eines pfändungs- una\nüberwe...,NaN,False,False,NaN,705e1764544006710cb46cac06c5686215b42653e20a29...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
4,e2482fb5-4d67-5836-8c87-39d0e17e105c,f3982f7e-3ea7-5abd-9212-298da881dcdd,Antrag auf Erlass eines Pfäll\n1\nÜberweisungs...,05.12.2023/1701792298_MP_001_05122023_165102.pdf,attachment_and_transfer_order,antrag auf erlass eines pfäll\nüberweisungsbes...,NaN,False,False,NaN,515ca20bf62b661992eedca7fc3424f9895622850b7cad...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1448,421c6599-37a0-5822-8d21-11de81eadee9,1ccc264b-566c-535f-a5aa-d2ea89e8547f,Bundeszentralamt\nfür Steuern\nPOSTANSCHRIFT\...,data/aftercourt/drittauskunf_with_invoice/comb...,drittauskunft,bundeszentralamt\nfür steuern\npostanschrift\n...,NaN,False,False,NaN,481d84aab52bb306dd926fb8952770634cac437de5d388...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nBundeszentralamt\nfür Steuern\nPOSTA...,True,False
1449,d9f6ba15-63cf-5743-aa24-12f26140ace4,8b976e45-a27a-5f35-8f19-9c04a4a315bb,Neue Büroanschrift ab 01.12.2024\nDortmunder ...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,neue büroanschrift ab 01.12.2024\ndortmunder s...,NaN,False,False,NaN,1d6062807bfb02abacfe2291ee4b68bcd1568d401b1acd...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nNeue Büroanschrift ab 01.12.2024\nDo...,False,False
1450,c72625b3-d66a-5df5-a933-11f57215838a,2a6e990a-3458-5efe-a8d9-f8b988af1740,Obergerichtsvollzieherin G. Samuels\nHESSEN\n...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,obergerichtsvollzieherin g. samuels\nhessen\nb...,NaN,False,False,NaN,6eb9335329883f671aa167608aa4108d45a5e0fcbbbb86...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nObergerichtsvollzieherin G. Samuels\...,False,False
1451,9b56d9ca-46f4-5711-ba70-0577a0a3bbbc,1366f284-dd3b-5940-af20-b7a3b07d0b62,F. Häußler\nMarktplatz 6\nGerichtsvollzieheri...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,f. häußler\nmarktplatz 6\ngerichtsvollzieherin...,NaN,False,False,NaN,0cb6fa6492e0550c6ec900e44ad20c4e1fed2355fd88b3...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nF. Häußler\nMarktplatz 6\nGerichtsvo...,False,False


In [90]:
# same changes to llm_input_sample
llm_input_sample.loc[llm_input_sample["attachment_id"].isin(not_fp_attachment_ids), "document_type"] = "va_dritt_invoice_protokol_combination"
llm_input_sample.loc[llm_input_sample["attachment_id"].isin(not_fp_attachment_ids), "is_ve_with_invoice"] = True
llm_input_sample.loc[llm_input_sample["attachment_id"].isin(not_fp_attachment_ids), "is_da_with_invoice"] = True
llm_input_sample.loc[llm_input_sample["attachment_id"].isin(not_fp_attachment_ids_no_pdf), "document_type"] = "drittauskunft"
llm_input_sample.loc[llm_input_sample["attachment_id"].isin(not_fp_attachment_ids_no_pdf), "is_ve_with_invoice"] = False

In [91]:
# Validate changes: compare before vs after (NaN-safe)
llm_input_orig = pd.read_csv(llm_input_sample_path)
diff_mask = (llm_input_sample != llm_input_orig) & ~(llm_input_sample.isna() & llm_input_orig.isna())
changed_mask = diff_mask.any(axis=1)
print(f"Total rows changed: {changed_mask.sum()}")
changed_cols = diff_mask.columns[diff_mask.any(axis=0)].tolist()
print(f"\nColumns changed: {changed_cols}")
print(f"\nPer-column change counts:")
for col in changed_cols:
    print(f"  {col}: {diff_mask[col].sum()} rows")
    
# Breakdown by group
mask1 = llm_input_orig["attachment_id"].isin(not_fp_attachment_ids)
mask2 = llm_input_orig["attachment_id"].isin(not_fp_attachment_ids_no_pdf)
print(f"\nGroup 1 (combination): {mask1.sum()} rows matched out of {len(not_fp_attachment_ids)} ids")
print(f"Group 2 (drittauskunft): {mask2.sum()} rows matched out of {len(not_fp_attachment_ids_no_pdf)} ids")

Total rows changed: 36

Columns changed: ['document_type', 'is_ve_with_invoice', 'is_da_with_invoice']

Per-column change counts:
  document_type: 36 rows
  is_ve_with_invoice: 36 rows
  is_da_with_invoice: 18 rows

Group 1 (combination): 18 rows matched out of 18 ids
Group 2 (drittauskunft): 18 rows matched out of 18 ids


In [92]:
# changes for predictions dataframes
pred_qwen32b = pd.read_csv(pred_qwen32b_path)
pred_qwen4b = pd.read_csv(pred_qwen4b_path)
pred_qwen35_4b = pd.read_csv(pred_qwen35_4b_path)

for name, df in [("pred_qwen32b", pred_qwen32b), ("pred_qwen4b", pred_qwen4b), ("pred_qwen35_4b", pred_qwen35_4b)]:
    df.loc[df["attachment_id"].isin(not_fp_attachment_ids), "document_type"] = "va_dritt_invoice_protokol_combination"
    df.loc[df["attachment_id"].isin(not_fp_attachment_ids_no_pdf), "document_type"] = "drittauskunft"

# Validate
for name, df, path in [
    ("pred_qwen32b", pred_qwen32b, pred_qwen32b_path),
    ("pred_qwen4b", pred_qwen4b, pred_qwen4b_path),
    ("pred_qwen35_4b", pred_qwen35_4b, pred_qwen35_4b_path),
]:
    orig = pd.read_csv(path)
    diff_mask = (df != orig) & ~(df.isna() & orig.isna())
    changed_mask = diff_mask.any(axis=1)
    changed_cols = diff_mask.columns[diff_mask.any(axis=0)].tolist()
    print(f"\n--- {name} ---")
    print(f"Total rows changed: {changed_mask.sum()}")
    print(f"Columns changed: {changed_cols}")
    for col in changed_cols:
        print(f"  {col}: {diff_mask[col].sum()} rows")
    mask1 = orig["attachment_id"].isin(not_fp_attachment_ids)
    mask2 = orig["attachment_id"].isin(not_fp_attachment_ids_no_pdf)
    print(f"Group 1 (combination): {mask1.sum()} matched out of {len(not_fp_attachment_ids)} ids")
    print(f"Group 2 (drittauskunft): {mask2.sum()} matched out of {len(not_fp_attachment_ids_no_pdf)} ids")


--- pred_qwen32b ---
Total rows changed: 36
Columns changed: ['document_type']
  document_type: 36 rows
Group 1 (combination): 18 matched out of 18 ids
Group 2 (drittauskunft): 18 matched out of 18 ids

--- pred_qwen4b ---
Total rows changed: 36
Columns changed: ['document_type']
  document_type: 36 rows
Group 1 (combination): 18 matched out of 18 ids
Group 2 (drittauskunft): 18 matched out of 18 ids

--- pred_qwen35_4b ---
Total rows changed: 36
Columns changed: ['document_type']
  document_type: 36 rows
Group 1 (combination): 18 matched out of 18 ids
Group 2 (drittauskunft): 18 matched out of 18 ids


In [ ]:
# # save changed dfs owerwrite
# raw_data.to_csv(raw_data_path, index=False)
# llm_input_sample.to_csv(llm_input_sample_path, index=False)
# pred_qwen32b.to_csv(pred_qwen32b_path, index=False)
# pred_qwen4b.to_csv(pred_qwen4b_path, index=False)
# pred_qwen35_4b.to_csv(pred_qwen35_4b_path, index=False)

Calculate the metrics again after changes

In [95]:
QWEN3_4B_DRITT_PATH= r"/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/analysis_data/llm_predictions/DATASET_ocr_MODEL_Qwen-4B_PROMPT_TARGET_drittauskunft_PROMPT_VERSION_V1_2026-06-01.csv"
QWEN35_4B_DRITT_PATH= r"/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/analysis_data/llm_predictions/DATASET_ocr_MODEL_Qwen-3.5-4B_PROMPT_TARGET_drittauskunft_PROMPT_VERSION_V1_2026-06-01.csv" 
QWEN3_32B_DRITT_PATH= r"/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/analysis_data/llm_predictions/DATASET_ocr_MODEL_Qwen-32B_PROMPT_TARGET_drittauskunft_PROMPT_VERSION_V1_2026-06-01.csv"

# read datasets
df_qwen3_4b_dritt = pd.read_csv(QWEN3_4B_DRITT_PATH)
df_qwen35_4b_dritt = pd.read_csv(QWEN35_4B_DRITT_PATH)
df_qwen3_32b_dritt = pd.read_csv(QWEN3_32B_DRITT_PATH)
df_qwen3_4b_dritt.rename(columns={"pred_llm_drittauskunft":"pred_q3_4b_dritt"}, inplace=True)
df_qwen3_32b_dritt.rename(columns={"pred_llm_drittauskunft":"pred_q3_32b_dritt"}, inplace=True)
df_qwen35_4b_dritt.rename(columns={"pred_llm_drittauskunft":"pred_q35_4b_dritt"}, inplace=True)
pred_data = df_qwen3_4b_dritt.merge(df_qwen3_32b_dritt[["ticket_uuid", "pred_q3_32b_dritt"]], on="ticket_uuid", how="inner")
pred_data = pred_data.merge(df_qwen35_4b_dritt[["ticket_uuid", "pred_q35_4b_dritt"]], on="ticket_uuid", how="inner")

pred_data["pred_json_q35_4b_dritt"] = pred_data["pred_q35_4b_dritt"].apply(extract_last_json)
pred_data["pred_json_q32b_dritt"] = pred_data["pred_q3_32b_dritt"].apply(extract_first_json)
pred_data["pred_json_q4b_dritt"] = pred_data["pred_q3_4b_dritt"].apply(extract_first_json)
pred_data = pred_data[pred_data["pred_json_q35_4b_dritt"] != {}].copy()
pred_data.document_type.value_counts()


format failed
format failed
format failed
format failed
format failed
format failed
format failed
format failed
format failed
{"output":{"is_dritt":false}
failed Expecting ',' delimiter: line 1 column 29 (char 28)
format failed
format failed
format failed
format failed
format failed
format failed
format failed
format failed


document_type
attachment_and_transfer_order             150
approved_seizure                          150
ladung_va                                 150
court_inbox                               150
monierung_mb                              150
mail_attachments                          150
fp_protocol                               143
fp_invoice                                101
drittauskunft                              70
vermögensverzeichnis                       70
approved_attachment_and_transfer_order     64
enforcement_order                          23
tbd                                        22
bailiff_ip                                 19
va_dritt_invoice_protokol_combination      18
contradiction                               3
neg_drittauskunft_hard                      1
Name: count, dtype: int64

In [96]:
# extract is_erlass and is_invoice from pred_json
pred_data["q3_4b_is_dritt"] = pred_data["pred_json_q4b_dritt"].apply(lambda x: x.get("is_dritt", None))
pred_data["q3_32b_is_dritt"] = pred_data["pred_json_q32b_dritt"].apply(lambda x: x.get("is_dritt", None))
pred_data["q35_4b_is_dritt"] = pred_data["pred_json_q35_4b_dritt"].apply(lambda x: x.get("is_dritt", None))


In [99]:
pred_data.document_type.value_counts()

document_type
attachment_and_transfer_order             150
approved_seizure                          150
ladung_va                                 150
court_inbox                               150
monierung_mb                              150
mail_attachments                          150
fp_protocol                               143
fp_invoice                                101
drittauskunft                              70
vermögensverzeichnis                       70
approved_attachment_and_transfer_order     64
enforcement_order                          23
tbd                                        22
bailiff_ip                                 19
va_dritt_invoice_protokol_combination      18
contradiction                               3
neg_drittauskunft_hard                      1
Name: count, dtype: int64

In [97]:
gt = pred_data["document_type"].apply(lambda x: True if x in ["drittauskunft","va_dritt_invoice_protokol_combination"] else False)
print(gt.value_counts())

document_type
False    1346
True       88
Name: count, dtype: int64


In [98]:
# calculate precision, recall, f1-score for each model
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

model_cols = {
    "Qwen3-4B": "q3_4b_is_dritt",
    "Qwen3-32B": "q3_32b_is_dritt",
    "Qwen3.5-4B": "q35_4b_is_dritt",
}

rows = []
for name, col in model_cols.items():
    pred = pred_data[col].fillna(False).astype(bool)
    rows.append({
        "model": name,
        "precision": precision_score(gt, pred, zero_division=0),
        "recall": recall_score(gt, pred, zero_division=0),
        "f1": f1_score(gt, pred, zero_division=0),
        "accuracy": accuracy_score(gt, pred),
    })

metrics_df = pd.DataFrame(rows).set_index("model")
metrics_df

,precision,recall,f1,accuracy
model,,,,
Qwen3-4B,0.488889,1.000000,0.656716,0.935844
Qwen3-32B,0.644444,0.988636,0.780269,0.965830
Qwen3.5-4B,0.639706,0.988636,0.776786,0.965132


In [104]:
print(metrics_df)

            precision    recall        f1  accuracy
model                                              
Qwen3-4B     0.488889  1.000000  0.656716  0.935844
Qwen3-32B    0.644444  0.988636  0.780269  0.965830
Qwen3.5-4B   0.639706  0.988636  0.776786  0.965132
